# Decorators - Stop Writing Repetetive Code

## What Problem Do Decorators Actually Solve?
You're building data pipelines. Every function needs:

- Logging (when did it run? what arguments?)
- Timing (is this slow? where's the bottleneck?)
- Error handling (retry failed API calls? catch and log exceptions?)
- Validation (are the inputs correct before processing?)

__Bad approach__: Copy-paste this code into every function. Result? 1000 lines of repetitive garbage.

__Good approach__: Write it ONCE as a decorator. Apply it with one line (@decorator_name).
That's it. Decorators let you add behavior to functions without cluttering the function itself. They're not magic—they're just functions that wrap other functions.

## Part 1: Functions Are Objects (You Need to Understand This First)
In Python, functions are just objects. You can pass them around like variables.

In [ ]:
def greet(name):
    return f"Hello, {name}"

# Assign to a variable
say_hello = greet
print(say_hello("Alice"))  # Works exactly like greet("Alice")

# Pass as an argument
def execute_twice(func, value):
    func(value)
    func(value)

execute_twice(print, "Hi")  # Prints "Hi" twice

__Why this matters__: Decorators work because you can pass functions to other functions. If you don't get this, decorators will feel like black magic.

## Part 2: Your First Decorator
Here is a decorator that prints "BEFORE" and "AFTER" around any function:

In [ ]:
def simple_decorator(func):
    def wrapper():
        print("BEFORE")
        func()
        print("AFTER")
    return wrapper

def say_hello():
    print("Hello!")

# Manual decoration
decorated = simple_decorator(say_hello)
decorated()
# Output:
# BEFORE
# Hello!
# AFTER

__What's happening?__

1. simple_decorator takes a function (func)
2. It defines a new function (wrapper) that runs code before/after func
3. It returns that new function
4. When you call decorated(), you're actually calling wrapper(), which calls the original func()

__The @ syntax is just shorthand__:

In [ ]:
@simple_decorator
def say_hello():
    print("Hello!")

# This is EXACTLY the same as:
# say_hello = simple_decorator(say_hello)

That is all @ does. It is not special, it is just cleaner.

## Part 3: Making Decorators Work With Any Function
Problem: The decorator above only works with functions that take no arguments. Try it with greet(name) and it breaks.

### Solution: Use *args and **kwargs

In [ ]:
def flexible_decorator(func):
    def wrapper(*args, **kwargs):
        print("BEFORE")
        result = func(*args, **kwargs)
        print("AFTER")
        return result
    return wrapper

@flexible_decorator
def add(a, b):
    return a + b

@flexible_decorator
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"

print(add(3, 5))        # Works
print(greet("Alice", greeting="Hi"))  # Works

__Rule__: Your wrapper should ALWAYS use *args, **kwargs unless you have a very specific reason not to.

## Part 4: Preserving Function Metadata (Use @wraps or You'll Regret It)

### Problem:

In [ ]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@my_decorator
def important_function():
    """This function does important stuff."""
    pass

print(important_function.__name__)  # "wrapper" (WRONG!)
print(important_function.__doc__)   # None (LOST THE DOCSTRING!)

Your function lost its name and docstring. This breaks debugging, documentation tools, and makes your code confusing.

### Solution: Use @wraps from functools

In [ ]:
from functools import wraps

def my_decorator(func):
    @wraps(func)  # ADD THIS LINE
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@my_decorator
def important_function():
    """This function does important stuff."""
    pass

print(important_function.__name__)  # "important_function" ✓
print(important_function.__doc__)   # "This function does..." ✓

__Rule__: Always use @wraps(func) on your wrapper. No exceptions.

## Part 5: Real Data Engineering Example - Timing
Lets build something useful. You need to know which parts of the ETL pipeline are slow.

In [ ]:
import time
from functools import wraps

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"{func.__name__} took {elapsed:.2f} seconds")
        return result
    return wrapper

@timer
def extract_data():
    time.sleep(2)     # Smulate API call
    return ["data1", "data2", "data3"]

@timer
def transform_data(data):
    time.sleep(1)     # Simulate processing
    return [d.upper() for d in data]

@timer
def load_data(data):
    time.sleep(0.5)    # Simulate database write
    return f"Load {len(data)} records"

data = extract_data()               # Prints: extract_data took 2.00 seconds
transformed = transform_data(data)  # Prints: transform_data took 1.00 seconds
result = load_data(transformed)     # Prints: load_data took 0.50 seconds

Now you instantly see where your bottlenecks are. No extra code in your functions.

## Part 6: Decorators with Parameters (Extra Level of Nesting)
Sometimes you need to configure your decorator. Example: retry a function N times.

In [ ]:
from functools import wraps
import time

def retry(max_attempts=3, delay=1):
    def decorator(func):  # Extra level!
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_attempts - 1:
                        raise
                    print(f"Attempt {attempt + 1} failed: {e}. Retrying in {delay}s...")
                    time.sleep(delay)
        return wrapper
    return decorator

@retry(max_attempts=3, delay=2)
def unstable_api_call():
    import random
    if random.random() < 0.7:  # 70% failure rate
        raise ConnectionError("API down")
    return {"status": "success"}

result = unstable_api_call()  # Auto-retries up to 3 times

### Structure for decorators with parameters:

In [ ]:
def decorator_with_params(param1, param2):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            # Use param1, param2 here
            return func(*args, **kwargs)
        return wrapper
    return decorator

Three levels: parameters → decorator → wrapper. Confusing at first, but you'll get used to it.

## Part 7: Stacking Decorators
You can apply multiple decorators. They stack bottom to top.

In [ ]:
@timer
@retry(max_attempts=3)
def fetch_data():
    # Code here
    pass

# Equivalent to:
# fetch_data = timer(retry(max_attempts=3)(fetch_data))

Bottom decorator wraps the function first, then the next one wraps that result.